# FaithfulMed Extractor Agent

This notebook is the working prototype for the Extractor agent. It loads MTSamples notes, splits the transcription into clinical sections, and extracts structured facts that later agents can use.

The goal is to convert raw transcribed medical text into a clean list of atomic facts such as diagnoses, medications, allergies, procedures, symptoms, and lab values.

In [1]:
# Set the path to the MTSamples CSV used for this extractor prototype.
# This lets the notebook find the data folder without depending on a specific local machine path.
from pathlib import Path
import csv
import json
import re

DATA_PATH = Path.cwd().parent / "data" / "mtsamples - mtsamples.csv"
print(DATA_PATH.exists())
print(DATA_PATH)

True
c:\Users\Samuel\pythonprojects\Tracking_software\Google-2E-faithfulmed-a-multi-agent-system-for-faithful-patient-friendly-medical-explanations\data\mtsamples - mtsamples.csv


In [2]:
# This function cleans the raw transcription text so the extraction step doesn't get stuck on
# extra spaces, commas, or punctuation from copied notes.
def clean_text(text: str) -> str:
    value = re.sub(r"\s+", " ", text or "").strip()
    value = re.sub(r"^[\s,;:.-]+|[\s,;:.-]+$", "", value)
    return value


# This helper removes common formatting artifacts from extracted values.
# For example, it strips leading commas and trailing periods that appear when we pull text
# directly from notes that are already formatted with labels like "ASSESSMENT:".
def normalize_fact_value(text: str) -> str:
    value = clean_text(text)
    value = re.sub(r"\s+([,.;:])", r"\1", value)
    value = re.sub(r"([,.;:]){2,}", ".", value)
    value = re.sub(r"\s+\.", ".", value)
    value = re.sub(r"\.+$", "", value)
    value = re.sub(r"^\.+", "", value)
    return clean_text(value)


# This function splits a clinical note into labeled sections such as SUBJECTIVE, MEDICATIONS,
# ALLERGIES, ASSESSMENT, and PLAN. It helps us parse the raw transcription in a structured way.
def get_sections(transcription: str) -> dict:
    if not transcription:
        return {}

    # Regex is used to detect section labels like "MEDICATIONS:" or "ASSESSMENT:".
    pattern = re.compile(
        r"(?i)\b([A-Z][A-Z /-]{2,}?)\s*:\s*|\b([A-Z][A-Z /-]{2,}?)\s*[,\n]"
    )
    matches = list(pattern.finditer(transcription))
    sections = {}

    # We take the text after each label and store it in a dictionary keyed by section name.
    for idx, match in enumerate(matches):
        label = (match.group(1) or match.group(2) or "").strip()
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(transcription)
        value = clean_text(transcription[start:end])
        if label:
            sections[label.upper()] = value

    # If the note does not use clear labels, we fall back to the whole note as one raw section.
    if not sections:
        sections["FULL_TEXT"] = clean_text(transcription)

    return sections


# This is the core Extractor function. It reads the note, grabs the relevant clinical sections,
# and turns them into a list of structured atoms that later stages can consume.
#
# The output follows a more medical-specific schema than the first draft, so the downstream
# Simplifier and Verifier will have cleaner, more usable inputs.
def extract_clinical_atoms(transcription: str) -> list[dict]:
    sections = get_sections(transcription)
    atoms = []

    # Helper to store one fact with a type, a cleaned value, and the source section it came from.
    def add_atom(atom_type: str, value: str, source: str = "", confidence: float = 1.0) -> None:
        if not value:
            return
        normalized_value = normalize_fact_value(value)
        if not normalized_value:
            return
        atoms.append({
            "type": atom_type,
            "value": normalized_value,
            "source": clean_text(source),
            "confidence": confidence,
        })

    # Diagnosis-related facts from assessment and impression sections.
    for key in ["ASSESSMENT", "IMPRESSION", "DIAGNOSIS", "POSTOPERATIVE DIAGNOSIS", "PREOPERATIVE DIAGNOSIS"]:
        if key in sections:
            add_atom("diagnosis", sections[key], key, confidence=0.95)

    # Medication facts from medication sections.
    for key in ["MEDICATIONS", "CURRENT MEDICATIONS", "HOME MEDICATIONS", "CURRENT MEDICATIONS:"]:
        if key in sections:
            add_atom("medication", sections[key], key, confidence=0.92)

    # Allergy facts from allergy sections.
    for key in ["ALLERGIES", "ALLERGIES TO MEDICATIONS"]:
        if key in sections:
            add_atom("allergy", sections[key], key, confidence=0.93)

    # Procedure and surgery facts.
    for key in ["PROCEDURE", "PROCEDURE PERFORMED", "OPERATION", "OPERATION PERFORMED", "PROCEDURES"]:
        if key in sections:
            add_atom("procedure", sections[key], key, confidence=0.9)

    # Clinical history and symptoms such as complaints or review-of-systems entries.
    for key in ["CHIEF COMPLAINT", "HISTORY OF PRESENT ILLNESS", "REVIEW OF SYSTEMS", "SUBJECTIVE"]:
        if key in sections:
            text = sections[key]
            if len(text) < 300:
                add_atom("symptom_or_history", text, key, confidence=0.88)

    # Lab values and measurement-like facts. This captures entries like "BP: 124/78".
    lab_pattern = re.compile(
        r"([A-Z][A-Za-z ]+\s*[:=]\s*[0-9]+\.?[0-9]*\s*(?:mg/dL|mmHg|%|mEq/L|mmol/L|units|cm|kg|lbs|bpm|C|F|mm)?)"
    )
    for field_name, value in sections.items():
        for match in lab_pattern.finditer(value):
            add_atom("lab_value", match.group(1), field_name, confidence=0.85)

    # These are clinical risk factors and comorbidities that often appear in history or PMH sections.
    risk_pattern = re.compile(r"\b(smoker|diabetes|hypertension|asthma|obesity|stroke|heart disease|sleep apnea)\b", flags=re.I)
    for field_name, value in sections.items():
        for match in risk_pattern.finditer(value):
            add_atom("risk_factor", match.group(0), field_name, confidence=0.8)

    # If nothing useful was extracted, fall back to the raw note to avoid empty output.
    if not atoms:
        add_atom("raw_text", transcription, "FULL_TEXT", confidence=0.5)

    return atoms


# This function loads the MTSamples CSV and returns each row as a Python dictionary.
# That allows us to iterate over real clinical notes without manually entering text.
def load_mtsamples(path: Path = DATA_PATH) -> list[dict]:
    with open(path, newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        return list(reader)

In [3]:
# This demo block loads the first MTSamples row, extracts atoms from its transcription, and prints
# the output. It is meant to validate that the pipeline works on a real clinical example.
rows = load_mtsamples()
sample = rows[0]
transcription = sample.get("transcription", "")
print(f"Loaded {len(rows)} MTSamples records.")
print("Sample specialty:", sample.get("medical_specialty", "Unknown"))
print("Sample name:", sample.get("sample_name", "Unknown"))
print("\nExtracted atoms from the first note:\n")
print(json.dumps(extract_clinical_atoms(transcription)[:8], indent=2, ensure_ascii=False))

Loaded 4999 MTSamples records.
Sample specialty: Allergy / Immunology
Sample name: Allergic Rhinitis

Extracted atoms from the first note:

[
  {
    "type": "diagnosis",
    "value": "Allergic rhinitis",
    "source": "ASSESSMENT",
    "confidence": 0.95
  },
  {
    "type": "medication",
    "value": "Her only medication currently is Ortho Tri-Cyclen and the Allegra",
    "source": "MEDICATIONS",
    "confidence": 0.92
  },
  {
    "type": "allergy",
    "value": "She has no known medicine allergies",
    "source": "ALLERGIES",
    "confidence": 0.93
  },
  {
    "type": "symptom_or_history",
    "value": "This 23-year-old white female presents with complaint of allergies. She used to have allergies when she lived in Seattle but she thinks they are worse here",
    "source": "SUBJECTIVE",
    "confidence": 0.88
  },
  {
    "type": "risk_factor",
    "value": "asthma",
    "source": "SHE HAS TRIED CLARITIN",
    "confidence": 0.8
  }
]
